In [3]:
import numpy as np
from typing import Any, Dict, Tuple

In [4]:
def batch_norm_forward(z: np.ndarray, gamma: np.ndarray, delta: np.ndarray, eps: float = 1e-5) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Compute the scalar batch normalization forward pass for a batch."""
    z = np.asarray(z, dtype=np.float64)
    gamma = np.asarray(gamma, dtype=np.float64)
    delta = np.asarray(delta, dtype=np.float64)
    mu = z.mean(axis=0, keepdims=True)
    centered = z - mu
    sq = centered ** 2
    var = sq.mean(axis=0, keepdims=True)
    std = np.sqrt(var + eps)
    inv_std = 1.0 / std
    normalized = centered * inv_std
    out = normalized * gamma + delta
    cache: Dict[str, Any] = {
        "z": z,
        "mu": mu,
        "centered": centered,
        "var": var,
        "std": std,
        "inv_std": inv_std,
        "normalized": normalized,
        "gamma": gamma,
        "gamma_shape": np.array(gamma).shape,
        "delta_shape": np.array(delta).shape,
    }
    return out, cache



def batch_norm_backward(dout: np.ndarray, cache: Dict[str, Any]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Backpropagate through the scalar batch normalization graph."""
    z = cache["z"]
    centered = cache["centered"]
    inv_std = cache["inv_std"]
    normalized = cache["normalized"]
    gamma = cache["gamma"]
    gamma_shape = cache["gamma_shape"]
    delta_shape = cache["delta_shape"]
    N = z.shape[0]
    dout = np.asarray(dout, dtype=np.float64)
    dgamma = np.sum(dout * normalized, axis=0)
    ddelta = np.sum(dout, axis=0)
    dxhat = dout * gamma
    dvar = np.sum(dxhat * centered * -0.5 * (inv_std ** 3), axis=0)
    dmu = np.sum(dxhat * -inv_std, axis=0) + dvar * np.mean(-2.0 * centered, axis=0)
    dz = dxhat * inv_std + dvar * 2.0 * centered / N + dmu / N
    dgamma = dgamma.reshape(gamma_shape) if gamma_shape else np.squeeze(dgamma)
    ddelta = ddelta.reshape(delta_shape) if delta_shape else np.squeeze(ddelta)
    return dz, dgamma, ddelta

In [5]:
def numerical_gradient_z(z: np.ndarray, gamma: np.ndarray, delta: np.ndarray, dout: np.ndarray, h: float = 1e-6) -> np.ndarray:
    grad = np.zeros_like(z, dtype=np.float64)
    it = np.nditer(z, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        idx = it.multi_index
        z_pos = z.copy()
        z_neg = z.copy()
        z_pos[idx] += h
        z_neg[idx] -= h
        out_pos, _ = batch_norm_forward(z_pos, gamma, delta)
        out_neg, _ = batch_norm_forward(z_neg, gamma, delta)
        grad[idx] = (np.sum(out_pos * dout) - np.sum(out_neg * dout)) / (2 * h)
        it.iternext()
    return grad


def numerical_gradient_param(param: np.ndarray, fun, h: float = 1e-6) -> np.ndarray:
    grad = np.zeros_like(param, dtype=np.float64)
    it = np.nditer(param, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        idx = it.multi_index
        param_pos = param.copy()
        param_neg = param.copy()
        param_pos[idx] += h
        param_neg[idx] -= h
        grad[idx] = (fun(param_pos) - fun(param_neg)) / (2 * h)
        it.iternext()
    return grad

In [6]:
rng = np.random.default_rng(42)
z = rng.normal(size=(5, 3))
gamma = rng.normal(size=(3,))
delta = rng.normal(size=(3,))
dout = rng.normal(size=(5, 3))
out, cache = batch_norm_forward(z, gamma, delta)
dz, dgamma, ddelta = batch_norm_backward(dout, cache)

num_dz = numerical_gradient_z(z, gamma, delta, dout)
loss_with_gamma = lambda g: np.sum(batch_norm_forward(z, g, delta)[0] * dout)
loss_with_delta = lambda d: np.sum(batch_norm_forward(z, gamma, d)[0] * dout)
num_dgamma = numerical_gradient_param(np.asarray(gamma), loss_with_gamma)
num_ddelta = numerical_gradient_param(np.asarray(delta), loss_with_delta)

print("matrix case")
print("max |dz - num_dz|:", np.max(np.abs(dz - num_dz)))
print("max |dgamma - num_dgamma|:", np.max(np.abs(dgamma - num_dgamma)))
print("max |ddelta - num_ddelta|:", np.max(np.abs(ddelta - num_ddelta)))

z1 = rng.normal(size=7)
gamma1 = np.array(0.9)
delta1 = np.array(-0.1)
dout1 = rng.normal(size=7)
_, cache1 = batch_norm_forward(z1, gamma1, delta1)
dz1, dgamma1, ddelta1 = batch_norm_backward(dout1, cache1)
num_dz1 = numerical_gradient_z(z1, gamma1, delta1, dout1)
loss_with_gamma1 = lambda g: np.sum(batch_norm_forward(z1, g, delta1)[0] * dout1)
loss_with_delta1 = lambda d: np.sum(batch_norm_forward(z1, gamma1, d)[0] * dout1)
num_dgamma1 = numerical_gradient_param(np.asarray(gamma1), loss_with_gamma1)
num_ddelta1 = numerical_gradient_param(np.asarray(delta1), loss_with_delta1)

print("vector case")
print("max |dz - num_dz|:", np.max(np.abs(dz1 - num_dz1)))
print("max |dgamma - num_dgamma|:", np.max(np.abs(dgamma1 - num_dgamma1)))
print("max |ddelta - num_ddelta|:", np.max(np.abs(ddelta1 - num_ddelta1)))

matrix case
max |dz - num_dz|: 9.863702077339553e-10
max |dgamma - num_dgamma|: 4.812769072159995e-10
max |ddelta - num_ddelta|: 6.446392308845361e-10
vector case
max |dz - num_dz|: 1.9533158424067665e-10
max |dgamma - num_dgamma|: 3.673372717116763e-11
max |ddelta - num_ddelta|: 3.5478286974921502e-12
